<h1 style="color:#1F4E78;">Flujo de efectivo</h1>

In [1]:
import pandas as pd
from openpyxl.styles import Font, Border, Side, Alignment, PatternFill
from openpyxl.utils import get_column_letter

df = pd.read_excel("Datos flujo de efectivo.xlsx")

df["inicio_semana"] = pd.to_datetime(df["inicio_semana"], dayfirst=True)

df["importe_flujo"] = df["importe"]
df.loc[df["tipo"] == "Egreso", "importe_flujo"] *= -1

flujo = (df.groupby(["categoría", "inicio_semana"])["importe_flujo"].sum().unstack(fill_value=0))

total_ingresos = (df[df["tipo"] == "Ingreso"].groupby("inicio_semana")["importe"].sum().reindex(flujo.columns, fill_value=0))
total_egresos = (df[df["tipo"] == "Egreso"].groupby("inicio_semana")["importe"].sum().reindex(flujo.columns, fill_value=0))

flujo_neto = total_ingresos - total_egresos

saldo_inicial = 0
saldo = saldo_inicial + flujo_neto.cumsum()

flujo.loc["TOTAL INGRESOS"] = total_ingresos
flujo.loc["TOTAL EGRESOS"] = -total_egresos
flujo.loc["FLUJO NETO"] = flujo_neto
flujo.loc["SALDO"] = saldo

nombres_semanas = (df[["inicio_semana", "semana"]].drop_duplicates().set_index("inicio_semana")["semana"].to_dict())

flujo.columns = [nombres_semanas[col] for col in flujo.columns]

flujo.index.name = "Categoría"

flujo = flujo.round(2)

archivo_salida = "Flujo de efectivo.xlsx"

with pd.ExcelWriter(archivo_salida, engine="openpyxl") as writer:
    flujo.to_excel(writer, sheet_name="Flujo")

    ws = writer.sheets["Flujo"]

    azul_encabezado = PatternFill(fill_type="solid", fgColor="1F4E78")
    azul_claro = PatternFill(fill_type="solid", fgColor="D9EAF7")
    azul_flujo = PatternFill(fill_type="solid", fgColor="9DC3E6")

    borde = Border(left=Side(style="thin"),right=Side(style="thin"),top=Side(style="thin"),bottom=Side(style="thin"))

    for row in ws.iter_rows():
        for cell in row:
            cell.border = borde
            cell.alignment = Alignment(horizontal="center", vertical="center")

    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = azul_encabezado

    for row in ws.iter_rows():
        if row[0].value in ["TOTAL INGRESOS", "TOTAL EGRESOS", "SALDO"]:
            for cell in row:
                cell.font = Font(bold=True)
                cell.fill = azul_claro

        if row[0].value == "FLUJO NETO":
            for cell in row:
                cell.font = Font(bold=True, size=13)
                cell.fill = azul_flujo

    for row in ws.iter_rows(min_row=2, min_col=2):
        for cell in row:
            cell.number_format = '#,##0.00'

    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)

        for cell in col:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = max_length + 2

print("Archivo creado:", archivo_salida)

print(flujo)

Archivo creado: Flujo de efectivo.xlsx
                   03-08 noviembre  10-15 noviembre  17-22 noviembre  \
Categoría                                                              
Bolsa                     -7700.00             0.00             0.00   
Caja                          0.00             0.00             0.00   
Certificación                 0.00             0.00             0.00   
Contador                  -8323.00         -9000.00         -9000.00   
Emplaye                       0.00             0.00             0.00   
Etiqueta                  -2532.45             0.00             0.00   
Gastos AVV               -25000.00             0.00             0.00   
Gastos CXM                    0.00             0.00             0.00   
Gastos LAR                    0.00             0.00             0.00   
Honorarios                    0.00             0.00             0.00   
Impuestos                     0.00             0.00             0.00   
Insumos                  